# Combine multiple OCEAN LoRAs (vLLM + baked combined adapter)

Each entry in `CONFIGS` is a dict mapping OCEAN slug (`o_plus`, `n_minus`, ...) to a scale.
For every config we **pre-bake** the requested adapters (scaled and summed) into a single PEFT adapter directory via `bake_combined_lora`, then drive **vLLM** with continuous batching across all questions. This is much faster than the previous PEFT-multi-adapter setup:
- one LoRA forward at inference time (combined), not N
- vLLM continuous-batches the whole question list per config
- adapters are cached on disk and only baked once per `(config, scratch dir)`

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

from src_dev.common.lora_catalogue import OCEAN_REGISTRY
from src_dev.utils.lora_combo_baking import bake_combined_lora

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Top-of-notebook config
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
QUESTIONS_DIR = Path("../data/ocean_open_ended")
BAKE_ROOT = Path("../scratch/combine_multiple_loras_baked")
N_QUESTIONS_PER_TRAIT = 3
MAX_NEW_TOKENS = 256

# Each config is a dict[slug, scale]; all listed adapters are summed (weighted) into one baked adapter.
CONFIGS: list[dict[str, float]] = [
    {"o_plus": 0.0, "c_plus": 0.0, "e_plus": 0.0, "a_plus": 0.0, "n_plus": 0.0},
    {"o_plus": 0.1, "c_plus": 0.1, "e_plus": 0.1, "a_plus": 0.1, "n_plus": 0.1},
    {"o_plus": 0.2, "c_plus": 0.2, "e_plus": 0.2, "a_plus": 0.2, "n_plus": 0.2},
    {"o_plus": 0.3, "c_plus": 0.3, "e_plus": 0.3, "a_plus": 0.3, "n_plus": 0.3},
    {"o_plus": 0.4, "c_plus": 0.4, "e_plus": 0.4, "a_plus": 0.4, "n_plus": 0.4},
    {"o_plus": 0.5, "c_plus": 0.5, "e_plus": 0.5, "a_plus": 0.5, "n_plus": 0.5},
]

In [ ]:
def load_questions(questions_dir: Path, n_per_trait: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    records: list[dict] = []
    for path in sorted(questions_dir.glob("*.jsonl")):
        with path.open() as f:
            rows = [json.loads(line) for line in f if line.strip()]
        sampled = rng.sample(rows, k=min(n_per_trait, len(rows)))
        records.extend(sampled)
    return records

questions = load_questions(QUESTIONS_DIR, N_QUESTIONS_PER_TRAIT, SEED)
print(f"Loaded {len(questions)} questions across {len(set(q['trait'] for q in questions))} traits")

In [ ]:
# Pre-bake one combined adapter per config.
# bake_combined_lora is idempotent on output_dir, so reruns are cheap.
# All-zero configs are treated as the base model (no LoRA at inference time).
BAKE_ROOT.mkdir(parents=True, exist_ok=True)

baked_configs: list[dict] = []
max_combined_rank = 0

for cfg_idx, scale_map in enumerate(CONFIGS):
    nonzero = {slug: float(s) for slug, s in scale_map.items() if float(s) != 0.0}
    if not nonzero:
        print(f"[cfg {cfg_idx}] all-zero -> base model (no bake)")
        baked_configs.append({"cfg_idx": cfg_idx, "scale_map": scale_map, "baked_path": None, "rank": 0})
        continue

    out_dir = BAKE_ROOT / f"cfg_{cfg_idx:02d}"
    pairs = [(OCEAN_REGISTRY[slug].adapter_ref, scale) for slug, scale in nonzero.items()]
    baked_path, combined_rank = bake_combined_lora(pairs, out_dir)
    max_combined_rank = max(max_combined_rank, combined_rank)
    print(f"[cfg {cfg_idx}] baked -> {baked_path}  (combined_rank={combined_rank})")
    baked_configs.append({"cfg_idx": cfg_idx, "scale_map": scale_map, "baked_path": baked_path, "rank": combined_rank})

print(f"\nmax_combined_rank across configs = {max_combined_rank}")

In [ ]:
# Initialize vLLM. max_lora_rank must be >= the largest combined rank across configs.
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

use_lora = max_combined_rank > 0
llm_kwargs = dict(
    model=BASE_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    enforce_eager=False,
)
if use_lora:
    llm_kwargs.update(
        enable_lora=True,
        max_loras=1,
        max_lora_rank=max(16, max_combined_rank),
    )

llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

In [ ]:
# Render all question prompts once via chat template (same for every config).
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": q["question"]}],
        add_generation_prompt=True,
        tokenize=False,
    )
    for q in questions
]

In [ ]:
# Main loop: one vLLM batched generate per config.
rows: list[dict] = []

for entry in baked_configs:
    cfg_idx = entry["cfg_idx"]
    scale_map = entry["scale_map"]
    baked_path = entry["baked_path"]

    print(f"\n=== Config {cfg_idx}: {scale_map} ===")
    if baked_path is None:
        outs = llm.generate(prompts, sampling_params)
    else:
        req = LoRARequest(
            lora_name=f"cfg{cfg_idx}",
            lora_int_id=cfg_idx + 1,
            lora_path=str(baked_path),
        )
        outs = llm.generate(prompts, sampling_params, lora_request=req)

    for q, out in zip(questions, outs):
        rows.append({
            "config_idx": cfg_idx,
            "config": scale_map,
            "trait": q["trait"],
            "facet": q["facet"],
            "question": q["question"],
            "response": out.outputs[0].text.strip(),
        })

df = pd.DataFrame(rows)
df

In [ ]:
pd.set_option("display.max_colwidth", None)
for idx, group in df.groupby("config_idx"):
    cfg = group["config"].iloc[0]
    print(f"\n========== Config {idx}: {cfg} ==========")
    for _, row in group.iterrows():
        print(f"\n[{row['trait']} / {row['facet']}] {row['question']}")
        print(f"  -> {row['response']}")

In [ ]:
# Clean up baked adapter dirs.
import shutil

if BAKE_ROOT.exists():
    shutil.rmtree(BAKE_ROOT)
    print(f"Removed {BAKE_ROOT}")